In [1]:
import argparse
import hail as hl
import os

WD = '/well/lindgren/UKBIOBANK/nbaya/wes_200k/ukb_wes_qc'
DATA_DIR  = f'{WD}/data'
RESOURCES_DIR = '/well/lindgren/UKBIOBANK/nbaya/resources'
ALL_CHR = list(map(str,range(1,23)))+['X','Y']

def hail_init(chrom=None, log_prefix='get_filtered_vcf'):
    r'''Initialize Hail
    '''
    n_slots = os.environ.get('NSLOTS', 1)
    chr_suffix = '' if chr is None else f'_chr{chrom}'
    hl.init(log=f'{WD}/logs/hail-{log_prefix}{chr_suffix}.log',
            default_reference='GRCh38',
            master=f'local[{n_slots}]')

def get_pre_gatk_path_prefix(chrom):
    r'''Return path prefix for pre-GATK annotation filtered MatrixTable/VCF
    '''
    return f'{DATA_DIR}/tmp/pre_gatk/tmp-ukb_wes_200k_split_filtered_pre_gatk_chr{chrom}'

def get_gatk_annot_path(chrom):
    r'''Get path to GATK-annotated sites-only VCF for chromosome `chrom`
    '''
    return f'{DATA_DIR}/tmp/gatk_annot/tmp-ukb_wes_200k_split_filtered_gatk_chr{chrom}.vcf.gz'

def get_filtered_mt_path_prefix(chrom):
    r'''Return path prefix for final filtered MatrixTable/VCF 
    '''
    return f'{DATA_DIR}/filtered/ukb_wes_200k_filtered_chr{chrom}'

def import_original_pvcf(chrom):
    r'''Import original OQFE pVCF for chromosome `chrom`
    '''
    pvcf_path = f'/well/ukbb-wes/pvcf/oqfe/ukbb-wes-oqfe-pvcf-chr{chrom}.vcf.gz'
    pvcf = hl.import_vcf(path=pvcf_path, force_bgz=True, array_elements_required=False)
    return pvcf

def import_original_bim(chrom):
    r'''Import original OQFE PLINK bim file for chromosome `chrom` as a Hail table
    '''
    bim_path = f'/well/ukbb-wes/calls/oqfe/ukbb-wes-oqfe-calls-chr{chrom}.bim'
    bim = hl.import_table(
        paths=bim_path, 
        no_header=True, 
        key=['f0','f3','f5','f4'],
        types={'f2':hl.tfloat,
               'f3':hl.tint})
    bim = bim.rename(
        {'f0':'chr',
         'f1':'rsid',
         'f2':'cm',
         'f3':'pos',
         'f4':'alt', # bim has alt allele first
         'f5':'ref'})
    return bim

def import_split_vcf(chrom):
    r'''Import VCF with multiallelics split and indels aligned with `bcftools norm`
    for chromosome `chrom`
    
    VCF to be used as input immediately before sample-,variant- and genotype-level QC
    '''
    split_path = f'{DATA_DIR}/tmp/split/tmp-ukb_wes_200k_split_chr{chrom}.vcf.gz'
    split = hl.import_vcf(path=split_path, force_bgz=True, array_elements_required=False)
    return split

def import_pre_gatk_vcf(chrom):
    r'''Import filtered pre-GATK annotation VCF for chromosome `chrom`
    '''
    pre_gatk_path = get_pre_gatk_path_prefix(chrom)+'.vcf.bgz'
    pre_gatk = hl.import_vcf(path=pre_gatk_path, force_bgz=True, array_elements_required=False)
    return pre_gatk

def import_gatk_annot(chrom):
    r'''Import GATK-annotated sites-only VCF for chromosome `chrom`
    '''
    annot_path = get_gatk_annot_path(chrom)
    annot = hl.import_vcf(path=annot_path, force_bgz=True, array_elements_required=False)
    annot = annot.rows() # only need rows, this is a sites-only VCF
    return annot

def count_vcf(vcf_path):
    r'''Imports VCF, prints path, row count and column count
    
    Printed statement is tab delimited
    '''
    mt = hl.import_vcf(path=vcf_path, force_bgz=True, array_elements_required=False)
    row_ct, col_ct = mt.count()
    print(f'{vcf_path}\t{row_ct}\t{col_ct}')

def write_post_qc_stats(chrom):
    r'''Writes post-QC sample and variant QC metrics to file
    '''        
    get_path = lambda name: f'{DATA_DIR}/stats/ukb_wes_200k_filtered_chr{chrom}.{name}.tsv'
    mt = hl.read_matrix_table(get_filtered_mt_path_prefix(chrom)+'.mt')
    
    names = ['sample_qc','variant_qc']
    names = [ n for n in names if not hl.hadoop_is_file(get_path(n))]
    
    if len(names)==0:
        print(f'Sample and variant QC metrics already written to:\n{" ".join(list(map(get_path, names)))}')
        return
    else:
        print(f'Running {" and ".join(names)} for chr{chrom}')
    
    qc_fns = [hl.sample_qc, hl.variant_qc]
    to_ht_fns = [hl.MatrixTable.cols, hl.MatrixTable.rows]
    flatten_structs = lambda ht, name: ht.annotate(**{name: ht[name].flatten()})
    hts_dict = dict(zip(names, [None]*len(names)))
    for name, qc_fn, to_ht_fn in zip(names, qc_fns, to_ht_fns):
        hts_dict[name] = flatten_structs(to_ht_fn(qc_fn(mt)).select(name), name)
    
    for name, ht in hts_dict.items():
        ht = ht.select(**{ f:v for f,v in ht[name].items() })
        if name=='variant_qc':
            ht = ht.annotate(**{ 'M'+f:hl.min(ht[f]) for f in ['AC','AF'] })
            ht = ht.transmute(**{ f:hl.delimit(ht[f]) for f in ['AC','AF','homozygote_count'] })
        
        ht.export(get_path(name))

def vcf_to_mt(path, out):
    r'''Write VCF with path=`path` to a MatrixTable with path=`out`
    '''
    mt = hl.import_vcf(path=path, force_bgz=True, array_elements_required=False)
    mt.write(out)

def get_fam(wes_200k_only=True):
    fam_path = f'{RESOURCES_DIR}/ukb12788_{"wes_200k_" if wes_200k_only else ""}pedigree.fam'
    fam = hl.import_table(paths=fam_path, key='IID',types={f:'str' for f in ['FID','IID','PAT','MAT','SEX','PHEN']})
    return fam

def filter_to_inliers(mt):
    r'''Filter to samples defined as inliers from the MAD thresholding
    '''
    path = f'{DATA_DIR}/sample_qc/ukb12788_all_inliers.custom_200k_consistent.sample_ids.txt'
    inliers = hl.import_table(path, no_header=True, key='f0')
    inliers = inliers.rename({'f0':'s'})
    mt = mt.filter_cols(hl.is_defined(inliers[mt.s]))
    return mt

def site_filter(mt):
    r'''Filter individual calls based on GQ, DP, allele balance filters
    mt (MatrixTable): MatrixTable to be filtered
    '''
    min_gq = 20 # exclude call if GQ<min_gq
    min_dp = 10 # exclude call if DP<min_dp
    snp_ab_pval = 1e-3
    indel_ab = 0.3
    sum_AD = hl.sum(mt.AD)
    mt = mt.filter_entries(
        hl.is_defined(mt.GT)
        & (
            (mt.GQ<min_gq)
            | (mt.DP<min_dp)
            ) 
        | ( mt.GT.is_het()
         & ( # SNP AB filter
            hl.is_snp(
                ref=mt.alleles[0],
                alt=mt.alleles[1]
            ) 
            & (
                  hl.binom_test(
                      x=mt.AD[1],
                      n=sum_AD,
                      p=0.5,
                      alternative='less') < snp_ab_pval
                  )
              ) 
         | ( # indel AB filter
                hl.is_indel(
                    ref=mt.alleles[0],
                    alt=mt.alleles[1]
                )
                & (mt.AD[1]/sum_AD < indel_ab)
              )
         ),
        keep = False
        )
    mt = recalc_info(mt)
    return mt

def in_target_expr(mt, padding : int = 0):
    r'''Returns BooleanExpression indicating whether a variant is in the sequencing target
    If locus overlaps with exome target or the padded target, then `in_target`=True
    or `in_target_{padding}bp`=True respectively.
    mt (MatrixTable): MatrixTable to be annotated/filtered
    padding (int): Base-pair padding to add to target regions, int
    remove (bool): If `remove`=True, this returns a MatrixTable that is filtered
        to rows where `in_target`=True
    '''
    target_path = f'{RESOURCES_DIR}/ref/xgen_plus_spikein.b38.chr_prefix.bed'
    target = hl.import_bed(target_path, reference_genome='GRCh38')
    if padding>0:
        start = target.interval.start
        end = target.interval.end
        padded_interval = hl.interval(hl.locus(contig=start.contig, pos=start.position-padding),
                                      hl.locus(contig=end.contig, pos=end.position+padding))
        padded_field = f'padded_interval_{padding}bp'
        target = target.annotate(**{padded_field : padded_interval})
        target = target.key_by(padded_field)
    
    return hl.is_defined(target[mt.locus])

def in_lcr_expr(mt):
    r'''Returns BooleanExpression indicating whether a variant is in a low-complexity region
    If locus is in low-complexity regions, then the expression is True, 
    otherwise the expression is False.
    
    Low-complexity regions are defined using supplementary data provided in
    Li 2014 (https://academic.oup.com/bioinformatics/article/30/20/2843/2422145)
    mt (MatrixTable): MatrixTable to be annotated/filtered
    remove (bool): If `remove`=True, this returns a MatrixTable that is filtered
        to rows where `in_lcr`=False, removing variants located in low-complexity
        regions
    '''
    lcr_path = f'{RESOURCES_DIR}/ref/LCR-hs38.bed.gz'
    lcr = hl.import_bed(lcr_path, reference_genome='GRCh38', force_bgz=True)
    
    return hl.is_defined(lcr[mt.locus])

def get_gnomadv3_pass(chrom=None):
    r'''Import VCFs containing whole-genome variants that have FILTER=PASS in gnomAD v3.1
    chrom (str, optional): If specified, the corresponding single-chrom VCF will be loaded.
        If not specified, the full, all-chromosomes VCF will be loaded.
    '''
    if not chrom is None:
        path = f'{RESOURCES_DIR}/gnomad/v3/gnomad.genomes.v3.1.sites.PASS.chr{chr}.vcf.bgz'
    elif (chrom is None) | ( chrom == 'all'):
        path = f'{RESOURCES_DIR}/gnomad/v3/gnomad.genomes.v3.1.sites.PASS.vcf.bgz'
    return hl.import_vcf(path=path, force_bgz=True, array_elements_required=False)

def in_gnomadv3_pass(mt, remove=False):
    r'''Creates row annotation `in_gnomadv3_pass`
    If locus overlaps with FILTER=PASS variants from gnomAD, then
    `in_gnomadv3_pass`=True, otherwise `in_gnomadv3_pass`=False
    mt (MatrixTable): MatrixTable to be filtered
    remove (bool): If `remove`=True, this returns a MatrixTable that is filtered
        to rows where `in_gnomadv3_pass`=True
    '''
    gnomad = get_gnomadv3_pass()
    mt = mt.annotate_rows(in_gnomadv3_pass = hl.is_defined(gnomad.rows()[mt.locus, mt.alleles]))
    if remove:
        mt = mt.filter_rows(mt.in_gnomadv3_pass)
    return mt

def get_ac(mt, min_ac=1):
    r'''Get allele counts for each biallelic variant from a VCF with multiallelic
    sites split
    min_ac (int): Minimum AC which `mt` is filtered to. If `min_ac`=0, then all
        variants are kept. If `min_ac`=1, all variants with AC>=1 are kept,
        i.e. all invariant sites (AC=0) are removed. Note that the AC field
        is always an array of length 1 because each variant is biallelic.
    '''
    path = f'{WD}/stats/ukb_wes_200k_inliers/ukb_wes_200k_inliers_split.vcf.gz'
    ac = hl.import_vcf(path=path, force_bgz=True, array_elements_required=False)
    assert ac.aggregate_rows(hl.agg.max(hl.len(ac.info.AC)))==1, 'The field `AC` must be an array of length 1 for all variants'
    mt = mt.annotate_rows(info = mt.info.annotate(AC = ac.rows()[mt.locus, mt.alleles].info.AC))
    if min_ac>0:
        mt = mt.filter_rows(mt.info.AC[0]>=min_ac)
    return mt

def get_n_alleles(max_alleles=None):
    pass # DEPRECATED now that we don't filter on number of alleles at a given site
    # r'''Creates INFO annotation `NALLELES` indicating the number of alleles at
    # a site before multi-allelic sites are split

    # mt (MatrixTable): MatrixTable to be annotated/filtered
    # max_alleles (int, optional): If not None, the returned MatrixTable will be
    #     filtered to rows with `allele_ct`<=`max_alleles`
    # '''
    # path = f'{WD}/vcf/ukb_wes_200k_inliers/ukb_wes_200k_inliers_sitesonly.vcf.gz'
    # presplit = hl.import_vcf(path=path, force_bgz=True, array_elements_required=False)
    # presplit = presplit.annotate_rows(info = presplit.info.annotate(NALLELES = presplit.alleles.length()))
    # out = f'{WD}/vcf/ukb_wes_200k_inliers/ukb_wes_200k_inliers_sitesonly_hail.vcf.bgz'
    # if not hl.hadoop_is_file(out):
    #     hl.export_vcf(presplit, out)
    # after this VCF is exported, run run_filter.sh with the "split_hail" filter
    # this will split multiallelics and align indels for this VCF

def excesshet_filter(mt, chrom):
    r'''Annotates rows of `mt` with GATK ExcessHet and returns `mt` filtered to 
    variants with ExcessHet less than or equal to `max_excesshet`.
    
    Excess heterozygosity is measured by GATK's ExcessHet annotation. 
    
    The `max_excesshet` threshold comes from the tutorial for VQSR:
        "The phred-scaled 54.69 corresponds to a z-score of -4.5." 
        https://gatk.broadinstitute.org/hc/en-us/articles/360035531112--How-to-Filter-variants-either-with-VQSR-or-by-hard-filtering
    '''    
    max_excesshet = 54.69 # ExcessHet threshold from GATK VQSR tutorial
    
    annot = import_gatk_annot(chrom)
    mt = mt.annotate_rows(info = mt.info.annotate(ExcessHet = annot[mt.row_key].info.ExcessHet))
    mt = mt.filter_rows(mt.info.ExcessHet <= max_excesshet)
    
    return mt

def pre_gatk_variant_filter(mt):
    r'''Filter VCF to variants that pass filters, in preparation for GATK annotation
    
    Previous versions of this filter included removing sites with a high number 
    of alleles. This is not included in this version because we think the 
    sample-, genotype- and these variant-level filters will do a decent job.
    '''
    # thresholds
    min_mac = 1 # minimum minor allele count (MAC) at a given site (where MAC=0 is an invariant site)
    padding = 50 # base pairs to pad around sequencing target regions
    
    # boolean filter expressions
    is_variant = hl.min(mt.info.AC)>=min_mac # True if site is variant (MAC>=1)
    is_in_target = in_target_expr(mt, padding=padding) # True if a variant is in the sequencing target region +/- padding base pairs
    is_in_lcr = in_lcr_expr(mt) # True if a variant is in a low-complexity region
    
    # filter mt
    mt = mt.filter_rows(
        is_variant
        & is_in_target
        & ~is_in_lcr)
    
    return mt

def post_gatk_variant_filter(mt, chrom):
    r'''Filter VCF to variants that pass filters, filtering on the GATK annotation "ExcessHet"
    '''    
    # annotate rows with ExcessHet and remove variants with ExcessHet>threshold
    mt = excesshet_filter(mt, chrom) 
    
    return mt

def recalc_info(mt, maf=None, info_field='info', gt_field='GT'):
    r'''Recalculate INFO fields AF, AC, AN and keep sites with minor allele 
    frequency > `maf`
    
    The fields AF and AC are made into integers instead of an array, only 
    showing the AF and AC for the alt allele
    '''
    if info_field in mt.row:
        mt = mt.annotate_rows(
            info = mt[info_field].annotate(**hl.agg.call_stats(mt[gt_field], 
                                                               mt.alleles)
                                               ).drop('homozygote_count')
            ) # recalculate AF, AC, AN, drop homozygote_count
    else:
        mt = mt.annotate_rows(**{info_field: hl.agg.call_stats(mt[gt_field], mt.alleles)})
    mt = mt.annotate_rows(
        **{info_field: mt[info_field].annotate(
            **{field:mt[info_field][field][1] for field in ['AF','AC']}
            )}
        ) # keep only the 2nd entries in AF and AC, which correspond to the alt alleles
    if maf is not None:
        mt = mt.filter_rows((mt[info_field].AF>maf) & (mt[info_field].AF<(1-maf)))
    return mt

def translate_sample_ids(mt, from_app: int, to_app: int):
    r'''Translate sample IDs from one UKB application to another
    
    `from_app` and `to_app` are UKB application IDs. This function only supports
    translation in either direction between applications 11867 (Lindgren) and 
    12788 (McVean)
    '''
    from_app, to_app = int(from_app), int(to_app)
    valid_apps = {11867, 12788}
    assert (from_app in valid_apps) & (to_app in valid_apps), f'from_app and to_app must be in {valid_apps}'
    assert from_app!=to_app, 'from_app and to_app must be different'
    print(f'Mapping IDs from UKB app {from_app} to {to_app}')
    id_dict = hl.import_table('/well/lindgren/UKBIOBANK/nbaya/resources/ukb11867_to_ukb12788.sample_ids.txt',
                              delimiter='\s+', key=f'eid_{from_app}')
    mt = mt.key_cols_by(s = id_dict[mt.s][f'eid_{to_app}'])
    undefined_ct = mt.aggregate_cols(hl.agg.sum(hl.is_missing(mt.s)))
    assert undefined_ct==0, f'Not all sample IDs mapped perfectly ({undefined_ct}/{mt.count_cols()} IDs are undefined)'
    return mt

def standardize_variant_ids(mt, chrom):
    r'''Replace existing rsid row expression with variant IDs from: 
        /well/ukbb-wes/calls/oqfe/ukbb-wes-oqfe-calls-chr*.bim
    with the format chr:pos:a1:a2, where:    
        chr : chromosome (no "chr" prefix; chromosomes X and Y are "23" and "24")
        pos : base pair position
        a1  : reference allele, unless variant is indel
        a2  : alternate allele, unless variant is indel
    If variant is an indel:
        a1  : "I" or "D", depending on whether variant is insertion or deletion, respectively
        a2  : number of base pairs inserted/deleted by indel
    For instance:
        ref=AGG  alt=A   ->  a2=2
        ref=T    alt=TT  ->  a2=1
    If >1 indels have the same "a1:a2" suffix, an additional suffix is added, 
    based on order of appearance, to make the variant IDs unique:
        chr1 962075 ref=CCCCCA alt=C -> 1:962075:I:5.01    
        chr1 962075 ref=CCCTCA alt=C -> 1:962075:I:5.02
    '''
    n_total = mt.count_rows()
    bim = import_original_bim(chrom)
    bim = bim.key_by(
        locus = hl.locus(
            contig = 'chr'+bim.chr.replace('23', 'X').replace('24','Y'),
            pos = bim.pos,
            reference_genome = 'GRCh38'),
        alleles = hl.array([bim.ref, bim.alt])
    )
    bim = bim.cache() # persist table in memory, this makes the next annotation faster
    mt = mt.annotate_rows(rsid = bim[mt.locus, mt.alleles].rsid)
    n_missing = mt.aggregate_rows(hl.agg.sum(hl.is_missing(mt.rsid)))
    assert n_missing==0, f'Some rsid values are missing ({n_missing}/{n_total} missing)'
    return mt

def get_vcf_metadata(info_fields_to_drop=[], format_fields_to_drop=['RNC'], path=None):
    r'''Get VCF metadata for exporting MatrixTable to VCF
    '''
    if path is None:
        path = '/well/ukbb-wes/pvcf/oqfe/ukbb-wes-oqfe-pvcf-chr1.vcf.gz' # use a VCF that has the same fields and is unlikely to be deleted
    metadata = hl.get_vcf_metadata(path)
    for f in info_fields_to_drop:
        metadata['info'].pop(f)
    for f in format_fields_to_drop:
        metadata['format'].pop(f)
    return metadata

def write(mt, out_prefix, write=False, checkpoint=False, export_vcf=False, export_plink=False,
          format_fields_to_drop=[], parallel=None, metadata = None):
    assert write or checkpoint or export_vcf or export_plink, 'MatrixTable will not be saved unless at least one of write_mt, export_to_vcf, or export_to_plink is True'
    if write and checkpoint:
        print('Warning: Both write=True and checkpoint=True. The write/read will be used instead of checkpoint.')
    
    mt = mt.drop(*[f for f in format_fields_to_drop if f in mt.entry])
    
    if write: # writing and reading back in takes more time than checkpointing, but takes up less disk space
        mt.write(out_prefix+'.mt')
        mt = hl.read_matrix_table(out_prefix+'.mt')
    
    if checkpoint:
        mt = mt.checkpoint(out_prefix+'.mt', _read_if_exists=True)
        mt = mt.drop(*[f for f in format_fields_to_drop if f in mt.entry]) # need to add this again because it may read an old existing file
        
    if export_vcf:
        out_vcf_path = out_prefix+('-tmp.vcf.bgz' if parallel is None else f'_{parallel}.bgz')
        if not hl.hadoop_is_file(out_vcf_path):
            if metadata is None:
                metadata = get_vcf_metadata(format_fields_to_drop=format_fields_to_drop)
            hl.export_vcf(dataset=mt,
                          output=out_vcf_path,
                          parallel=parallel,
                          metadata = metadata)
        else:
            raise ValueError(f'Cannot export VCF because {out_vcf_path} already exists')
    
    if export_plink:
        if not any(hl.hadoop_is_file(out_prefix+s) for s in ['bed','bim','fam']): # don't overwrite if any files exist
            hl.export_plink(dataset=mt,
                            output=out_prefix)
        else:
            raise ValueError(f'Cannot export to PLINK because at least one of {out_prefix}.{{bed,bim,fam}} already exists')


In [2]:
hail_init(22)

Running on Apache Spark version 3.1.2
SparkUI available at http://compe066.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.77-684f32d73643
LOGGING: writing to /well/lindgren/UKBIOBANK/nbaya/wes_200k/ukb_wes_qc/logs/hail-get_filtered_vcf_chr22.log


In [5]:
## tmp files are missing
#mt = import_split_vcf(22)
#mt = filter_to_inliers(mt)
#mt = site_filter(mt)

In [6]:
out_prefix = get_pre_gatk_path_prefix(22)

In [7]:
mt = hl.read_matrix_table(f'{out_prefix}.mt')

FatalError: HailException: No file or directory found at /well/lindgren/UKBIOBANK/nbaya/wes_200k/ukb_wes_qc/data/tmp/pre_gatk/tmp-ukb_wes_200k_split_filtered_pre_gatk_chr22.mt

Java stack trace:
is.hail.utils.HailException: No file or directory found at /well/lindgren/UKBIOBANK/nbaya/wes_200k/ukb_wes_qc/data/tmp/pre_gatk/tmp-ukb_wes_200k_split_filtered_pre_gatk_chr22.mt
	at is.hail.utils.ErrorHandling.fatal(ErrorHandling.scala:11)
	at is.hail.utils.ErrorHandling.fatal$(ErrorHandling.scala:11)
	at is.hail.utils.package$.fatal(package.scala:78)
	at is.hail.expr.ir.RelationalSpec$.readMetadata(AbstractMatrixTableSpec.scala:32)
	at is.hail.expr.ir.RelationalSpec$.readReferences(AbstractMatrixTableSpec.scala:73)
	at is.hail.variant.ReferenceGenome$.fromHailDataset(ReferenceGenome.scala:581)
	at is.hail.variant.ReferenceGenome.fromHailDataset(ReferenceGenome.scala)
	at sun.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at sun.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAcce